In [1]:
!pip install beautifulsoup4

In [2]:
!pip install Selenium

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 37.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 460.2/460.2 kB 26.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.8 MB/s eta 0:00:00


In [3]:
# 코랩을 시작할 때 아래코드를 한 번 돌려줍니다.
!apt-get update
!apt install chromium-chromedriver
!cp /usr/lib/chromium-browser/chromedriver /usr/bin

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [110 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [119 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [109 kB]
Hit:7 https://ppa.launchpadcontent.net/c2d4u.team/c2d4u4.0+/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,054 kB]
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [662 kB]
Get:13 http://security.ubu

In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import pandas as pd
from selenium.webdriver.common.by import By
import requests
from bs4 import BeautifulSoup

import time
import datetime

In [5]:
# Chrome 옵션 설정
options = Options()
options.add_argument('--headless')  # 헤드리스 모드 설정
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

# ChromeDriver 실행
driver = webdriver.Chrome(options=options)


# 써클차트 digital 음원 크롤링

In [6]:
# 크롬 드라이버 초기화
driver = webdriver.Chrome(options=options)


year = 2018
url = f"https://circlechart.kr/page_chart/onoff.circle?nationGbn=T&serviceGbn=ALL&targetTime={year}&hitYear={year}&termGbn=year&yearTime=3"

driver.get(url)


#time.sleep(5)

# 페이지의 HTML을 가져와 BeautifulSoup으로 파싱합니다.
soup = BeautifulSoup(driver.page_source, 'html.parser')

# #mo_chart_tbody 아래의 모든 tr 요소를 찾습니다.
rows = soup.select('#mo_chart_tbody tr')



In [7]:
# 위에서 얻은 데이터를 DataFrame으로 변환
data = []

for index, row in enumerate(rows, start=1):
    title_selector = f'td:nth-child(2) > div > section:nth-child(2) > div > div:nth-child(1)'
    artist_selector = f'td:nth-child(2) > div > section:nth-child(2) > div > div.text-sm.text-gray-400'
    score_selector = f'td:nth-child(2) > div > section:nth-child(2) > div > div:nth-child(3)'

    title = row.select_one(title_selector).text.strip()
    artist = row.select_one(artist_selector).text.split('|')[0].strip()
    score = row.select_one(score_selector).text.strip()

    data.append({
        "순위": index,
        "노래 제목": title,
        "가수": artist,
        "스코어": score
    })

# 드라이버 종료
driver.quit()

In [8]:
# 데이터프레임 생성
df = pd.DataFrame(data)

# Excel 파일로 저장
df.to_excel("2018년연간음원.xlsx", index=False)

# 써클차트 음반 크롤링

In [9]:
# ChromeDriver 초기화
driver = webdriver.Chrome(options=options)


year = 2018
url = f'https://circlechart.kr/page_chart/album.circle?nationGbn=T&targetTime={year}&hitYear={year}&termGbn=year&yearTime=3'

driver.get(url)

# BeautifulSoup으로 HTML 파싱
soup = BeautifulSoup(driver.page_source, 'html.parser')

# 각 행을 선택
rows = soup.select('#pc_chart_tbody tr')

In [10]:
# 위에서 얻은 데이터를 DataFrame으로 변환
data = []

for index, row in enumerate(rows, start=1):
    title_selector = 'td:nth-child(3) div.font-bold.mb-2'
    artist_selector = 'td:nth-child(3) div.text-sm.text-gray-400.font-bold'
    sales_selector = 'td.text-center span.font-bold'

    title = row.select_one(title_selector).text.strip()
    artist = row.select_one(artist_selector).text.strip()
    sales = row.select_one(sales_selector).text.strip()


    data.append({
        "순위": index,
        "노래 제목": title,
        "가수": artist,
        "음반 판매량": sales
    })

# 드라이버 종료
driver.quit()

In [11]:
# 데이터프레임 생성
df = pd.DataFrame(data)

# Excel 파일로 저장
df.to_excel("2018년연간음반.xlsx", index=False)